# S3.17 - collect_list, collect_set
**Date completed:** October 2026  
**Status:** Completed  
**Dataset:** samples.tpch.orders, 7.5M rows \
**Key concept:** collect_list and collect_set gather many rows into one array per group (the reverse of explode)

In [0]:
# ============================================================
# S3.17 — Cell 2
# collect_list(), collect_set() and Array Aggregation
#
# Goal:
# 1. Understand collect_list() vs collect_set().
# 2. Observe duplicate and NULL behaviour.
# 3. Understand non-deterministic array ordering after grouping.
# 4. Use sort_array() when stable value ordering is required.
# 5. Validate array sizes against count() semantics.
# 6. Apply array aggregation to real TPC-H orders.
# 7. Understand collect_list() → explode() round-trip limits.
# 8. Filter aggregated arrays with array_contains().
# 9. Understand memory/performance implications of collecting
#    many values into one group.
#
# Dataset:
# Controlled sample data
# samples.tpch.orders
#
# Important:
#
# collect_list()
# → keeps duplicate NON-NULL values
#
# collect_set()
# → keeps distinct NON-NULL values
#
# Neither guarantees element ordering after shuffle.
# ============================================================

import pyspark.sql.functions as F


print("=== S3.17: collect_list() AND collect_set() ===")
print()


# ------------------------------------------------------------
# Step 1 — Controlled Small Dataset
#
# Customer 1:
# F, F, O
#
# Customer 2:
# P, P
#
# Customer 3:
# F, NULL, O
#
# This lets us observe:
#
# duplicates
# NULL behaviour
# list vs set behaviour
# ------------------------------------------------------------

small_data = [

    (1, "F"),
    (1, "F"),
    (1, "O"),

    (2, "P"),
    (2, "P"),

    (3, "F"),
    (3, None),
    (3, "O"),
]


df_small = spark.createDataFrame(
    small_data,
    [
        "customer_id",
        "status"
    ]
)

print("=== STEP 1: SOURCE DATA ===")

df_small.show(
    truncate=False
)

print()



In [0]:
# ------------------------------------------------------------
# Step 2 — collect_list() vs collect_set()
#
# collect_list(status):
#
# → keeps duplicate non-NULL values
#
# collect_set(status):
#
# → keeps distinct non-NULL values
#
# Both ignore NULL input values.
#
# Important count comparison:
#
# count("*")
# → counts every row
#
# count("status")
# → counts only non-NULL status values
#
# size(collect_list("status"))
# → should therefore match count("status")
# ------------------------------------------------------------

print(
    "=== STEP 2: collect_list() vs collect_set() ==="
)


df_small_agg = (

    df_small

    .groupBy(
        "customer_id"
    )

    .agg(

        F.collect_list("status").alias("status_list"),

        F.collect_set("status").alias("status_set"),

        F.count("*").alias("rows_in_group"),

        F.count("status").alias("non_null_status_count"))

    .orderBy("customer_id"))

df_small_agg.show(
    truncate=False
)

print()


print("""
Expected behaviour:

Customer 1:

Input:
F, F, O

collect_list:
[F, F, O]

collect_set:
[F, O]


Customer 2:

Input:
P, P

collect_list:
[P, P]

collect_set:
[P]


Customer 3:

Input:
F, NULL, O

collect_list:
[F, O]

collect_set:
[F, O]

The NULL value is not collected.
""")

In [0]:
# ------------------------------------------------------------
# Step 3 — Validate Collection Sizes
#
# Important relationship:
#
# size(collect_list(status))
# =
# count(status)
#
# NOT necessarily:
#
# size(collect_list(status))
# =
# count("*")
#
# because NULL values are excluded from collect_list().
# ------------------------------------------------------------

print(
    "=== STEP 3: COLLECTION SIZE VALIDATION ==="
)


df_size_check = (

    df_small

    .groupBy(
        "customer_id"
    )

    .agg(

        F.size(F.collect_list("status")).alias("list_size"),

        F.size(F.collect_set("status")).alias("set_size"),

        F.count("*").alias("row_count"),

        F.count("status").alias("non_null_status_count"))

    .orderBy(
        "customer_id"
    )
)

df_size_check.show(
    truncate=False
)

invalid_list_sizes = (

    df_size_check

    .filter(F.col("list_size") !=  F.col("non_null_status_count")).count())

assert invalid_list_sizes == 0, (
    "collect_list size should match count(status)"
)

print(
    "collect_list size validation passed.")

print()


In [0]:
# ------------------------------------------------------------
# Step 4 — Array Ordering
#
# collect_list() and collect_set() do NOT guarantee
# deterministic element ordering after a shuffle.
#
# Example:
#
# [F, O, P]
#
# could potentially be returned in another order.
#
#
# sort_array():
#
# sorts the collected values by value.
#
# This makes the displayed array stable according to the
# sort order.
#
#
# Important:
#
# sort_array() does NOT restore original source-row order.
#
# It simply sorts the resulting array values.
# ------------------------------------------------------------

print(
    "=== STEP 4: SORTED ARRAY OUTPUT ==="
)


df_small_sorted = (

    df_small

    .groupBy("customer_id")

    .agg(

        F.sort_array(F.collect_list("status")).alias("status_list_sorted"),

        F.sort_array(F.collect_set("status")).alias("status_set_sorted"))

    .orderBy("customer_id"))

df_small_sorted.show(
    truncate=False
)

print()


In [0]:
# ------------------------------------------------------------
# Step 5 — Real TPC-H Data
#
# Use customers already investigated in S3.16.
#
# Target grain:
#
# ONE ROW PER CUSTOMER
#
# Validate:
#
# list_size
# =
# count(o_orderstatus)
#
# If o_orderstatus is non-NULL for every order:
#
# count(o_orderstatus)
# =
# count(*)
#
# and therefore list_size == order_count.
# ------------------------------------------------------------

print(
    "=== STEP 5: REAL TPC-H DATA ==="
)


df_orders = spark.table(
    "samples.tpch.orders"
)


known_customers = [
    721180,
    382414,
    299701
]


df_real = (

    df_orders

    .filter(F.col("o_custkey").isin(known_customers))

    .groupBy("o_custkey")

    .agg(

        F.sort_array(F.collect_set("o_orderstatus")).alias("distinct_statuses"),

        F.size(F.collect_list("o_orderstatus")).alias("list_size"),

        F.size(F.collect_set("o_orderstatus")).alias("set_size"),

        F.count("*").alias("order_count"),

        F.count("o_orderstatus").alias("non_null_status_count"))

    .orderBy(
        "o_custkey"
    )
)

df_real.show(
    truncate=False
)

print()



In [0]:
# ------------------------------------------------------------
# Step 6 — Validate Real-Data Array Sizes
# ------------------------------------------------------------

print(
    "=== STEP 6: REAL-DATA VALIDATION ==="
)


invalid_real_sizes = (

    df_real

    .filter(F.col("list_size") != F.col("non_null_status_count")).count())

assert invalid_real_sizes == 0, ("Real-data collect_list size validation failed")

print("list_size == non_null_status_count validation passed.")

# Check whether TPCH statuses are NULL for these customers.

null_status_rows = (

    df_orders

    .filter(F.col("o_custkey").isin(known_customers))

    .filter(F.col("o_orderstatus").isNull()).count())

print(
    f"NULL status rows for known customers: "
    f"{null_status_rows}")

if null_status_rows == 0:

    print(
        "Therefore list_size also equals order_count "
        "for this sample."
    )

print()

In [0]:
# ------------------------------------------------------------
# Step 7 — Connection to S3.16
#
# Customer 721180 had:
#
# Fulfilled orders = 15
# Open orders      = 20
# Pending orders   = 3
#
# Total            = 38
#
#
# collect_set(status)
#
# should therefore show:
#
# [F, O, P]
#
# and:
#
# list_size
#
# should be 38 because o_orderstatus is non-NULL
# for this customer's orders.
# ------------------------------------------------------------

print(
    "=== STEP 7: S3.16 CROSS-CHECK ==="
)


customer_721180 = (

    df_real

    .filter(F.col("o_custkey") == 721180))

customer_721180.show(truncate=False)

print()


In [0]:
# ------------------------------------------------------------
# Step 8 — collect_list() → explode() Round Trip
#
# We collect:
#
# o_orderkey
#
# for customer 721180.
#
#
# Important:
#
# o_orderkey is non-NULL here.
#
# Therefore:
#
# collect_list(orderkey)
#     ↓
# one array
#     ↓
# explode(array)
#     ↓
# same number of order-key values
#
#
# But this is NOT a universal exact row round-trip.
#
# Why?
#
# 1. NULL collection values would be dropped.
# 2. Original row order is not preserved.
# 3. We only collected one column.
#
# We are recovering the collected order-key values,
# not reconstructing the complete original DataFrame.
# ------------------------------------------------------------

print(
    "=== STEP 8: collect_list() → explode() ==="
)


df_original_721180 = (

    df_orders

    .filter(F.col("o_custkey") == 721180)

    .select(
        "o_custkey",
        "o_orderkey"))


original_count = (
    df_original_721180.count())

df_one = (

    df_original_721180

    .groupBy("o_custkey")

    .agg(

        F.collect_list("o_orderkey").alias("order_keys")))


df_back = (

    df_one

    .select(

        "o_custkey",

        F.explode("order_keys").alias("o_orderkey")))


collected_group_count = (df_one.count())


exploded_count = (df_back.count())


print(f"Original order rows   : {original_count}")

print(f"Grouped customer rows : {collected_group_count}")

print(f"Rows after explode    : {exploded_count}")

assert original_count == exploded_count, ("Round-trip count validation failed")


print("Non-NULL order-key count round trip passed.")

print()

In [0]:
# ------------------------------------------------------------
# Step 9 — Validate Round-Trip Values
#
# Count equality alone is not enough.
#
# We also verify that exploded order keys match the original
# customer order keys.
# ------------------------------------------------------------

print(
    "=== STEP 9: ROUND-TRIP VALUE VALIDATION ==="
)


missing_after_roundtrip = (

    df_original_721180

    .join(df_back,on=["o_custkey", "o_orderkey"],how="left_anti").count())

extra_after_roundtrip = (

    df_back

    .join(df_original_721180,on=["o_custkey","o_orderkey"],how="left_anti").count())

print(
    f"Missing values after round trip : "
    f"{missing_after_roundtrip}")

print(
    f"Extra values after round trip   : "
    f"{extra_after_roundtrip}")

assert missing_after_roundtrip == 0

assert extra_after_roundtrip == 0


print(
    "Round-trip value validation passed.")

print()

In [0]:
# ------------------------------------------------------------
# Step 10 — Filter an Array Column
#
# array_contains(array, value)
#
# checks whether an array contains a value.
#
#
# Requirement:
#
# Keep customers that have at least one Pending order.
#
# Because distinct_statuses was created using:
#
# collect_set(o_orderstatus)
#
# array_contains(..., "P")
#
# answers:
#
# Does this customer have status P?
# ------------------------------------------------------------

print(
    "=== STEP 10: array_contains() ==="
)

df_has_p = (

    df_real

    .filter(

        F.array_contains(F.col("distinct_statuses"),"P")))

df_has_p.select(
    "o_custkey",
    "distinct_statuses").show(truncate=False)

print()


In [0]:
# ------------------------------------------------------------
# Step 11 — collect_list() vs collect_set()
# ------------------------------------------------------------

print(
    "=== STEP 11: COMPARISON ==="
)

print("""
collect_list(column)

→ Returns ArrayType.
→ Keeps duplicate non-NULL values.
→ Ignores NULL input values.
→ Element order is not guaranteed after shuffle.


collect_set(column)

→ Returns ArrayType.
→ Removes duplicate values.
→ Ignores NULL input values.
→ Element order is not guaranteed.


sort_array(...)

→ Sorts array elements by value.
→ Useful for stable display/comparison.
→ Does NOT recover original source order.


size(array)

→ Number of elements in an array.


array_contains(array, value)

→ Tests whether an array contains a value.


explode(array)

→ Converts array elements back into separate rows.
""")

In [0]:
# ------------------------------------------------------------
# Step 12 — Performance / Memory Awareness
#
# collect_list() and collect_set() are Spark aggregations.
#
# Do NOT confuse:
#
# collect_list()
#
# with:
#
# DataFrame.collect()
#
#
# collect_list():
#
# Creates an ARRAY inside each GROUP.
#
#
# DataFrame.collect():
#
# Returns ALL DataFrame rows to the DRIVER.
#
#
# However:
#
# collect_list() / collect_set() can still create very large
# aggregation states if a group contains many values.
#
# Example:
#
# groupBy(country)
# collect_list(transaction_id)
#
# If one country contains millions of transactions,
# that group's array can become extremely large.
# ------------------------------------------------------------

print(
    "=== STEP 12: PERFORMANCE AWARENESS ==="
)


print("""
collect_list()
and
collect_set()

are distributed aggregation functions.

They do NOT mean the same thing as:

DataFrame.collect()


collect_list():

Rows
  ↓
groupBy(key)
  ↓
Array stored inside each group


DataFrame.collect():

Distributed DataFrame
  ↓
ALL result rows
  ↓
Driver


Important:

Very large groups can produce very large arrays.

Use collect_list() / collect_set() only when creating the
array is meaningful for the business requirement.
""")

print("=== S3.17 CELL 2 COMPLETED ===")

In [0]:
# ------------------------------------------------------------
# Step 13 — Final Learning Summary
# ------------------------------------------------------------

print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
collect_list():

Keeps duplicate non-NULL values.

Example:

F, F, O

→ [F, F, O]


collect_set():

Keeps distinct non-NULL values.

Example:

F, F, O

→ [F, O]


NULL:

Both functions ignore NULL input values.


ORDER:

Neither function guarantees array element order
after aggregation/shuffle.


sort_array():

Use when stable value ordering is required.

It sorts by value.

It does NOT restore original input-row ordering.


COUNT RELATIONSHIP:

size(
    collect_list(column)
)

=

count(column)


Not necessarily:

count("*")

because NULL column values are not collected.


ROUND TRIP:

collect_list(non_null_column)
        ↓
explode()
        ↓
recovers collected values and multiplicity

but:

- NULL collection values are lost
- original ordering is not preserved
- uncollected columns are not reconstructed


ARRAY FILTER:

array_contains(
    array_column,
    value
)

→ tests whether value exists in the array.


PERFORMANCE:

collect_list() / collect_set()
can create large arrays for high-cardinality groups.

Only aggregate values into arrays when that output structure
is actually required.
""")


# S3.17 — Key Takeaways: collect_list() and collect_set()

## 1. What They Do

Both are aggregation functions that convert multiple rows within a group into an array.

```text
Many rows
    ↓
groupBy(key)
    ↓
collect_list() / collect_set()
    ↓
One row containing an array
```

### collect_list()

```python
F.collect_list("status")
```

- Keeps duplicate non-NULL values.
- Returns `ArrayType`.

Example:

```text
F, F, O
→ [F, F, O]
```

### collect_set()

```python
F.collect_set("status")
```

- Removes duplicate values.
- Returns `ArrayType`, not a Python `set`.

Example:

```text
F, F, O
→ [F, O]
```

---

## 2. NULL Behaviour

Both functions skip NULL input values.

Example:

```text
Input:

F
NULL
O
```

Results:

```text
collect_list(status)
→ [F, O]

collect_set(status)
→ [F, O]
```

Therefore:

```text
size(collect_list(col))
=
count(col)
```

because both consider non-NULL values.

But this is not necessarily equal to:

```text
count("*")
```

because `count("*")` counts every row, including rows where that column is NULL.

---

## 3. list_size vs set_size

```python
F.size(
    F.collect_list("status")
)
```

returns the number of collected non-NULL values, including duplicates.

```python
F.size(
    F.collect_set("status")
)
```

returns the number of distinct collected values.

Example:

```text
Input:
F, F, O, P

list_size
→ 4

set_size
→ 3
```

---

## 4. Element Ordering

Neither `collect_list()` nor `collect_set()` guarantees element order after aggregation/shuffle.

Do not assume:

```text
[F, O, P]
```

will always naturally appear in that order.

For stable value ordering:

```python
F.sort_array(
    F.collect_set("status")
)
```

### Important

`sort_array()` sorts by value.

It does **not** restore the original source-row order.

---

## 5. Relationship with explode()

Conceptually:

```text
explode()
Array
→ many rows
```

while:

```text
collect_list()
many rows
→ Array
```

They can therefore look like opposite operations, but they are not exact inverses.

Example:

```text
Rows
1
2
3

↓ collect_list()

[1, 2, 3]

↓ explode()

1
2
3
```

However, the round trip has limitations.

---

## 6. Round-Trip Limitations

```text
collect_list()
    ↓
explode()
```

can recover the collected values and their multiplicity.

But:

- NULL collection values are lost.
- Original row ordering is not preserved.
- Columns that were not collected are not reconstructed.
- It only reproduces the values placed inside the array.

Therefore:

```text
collect_list → explode
```

is a useful value-level round trip, not a guaranteed reconstruction of the original DataFrame.

---

## 7. Our Round-Trip Result

For customer `721180`:

```text
Original rows         = 38
Grouped customer rows = 1
Rows after explode    = 38
```

So:

```text
38 → 1 array row → 38 exploded rows
```

The counts matched because the collected `o_orderkey` values were non-NULL.

---

## 8. Filtering Array Columns

Use:

```python
F.array_contains(
    F.col("array_column"),
    value
)
```

Example:

```python
F.array_contains(
    F.col("status_set"),
    "P"
)
```

asks:

```text
Does this array contain P?
```

---

## 9. Small-Data Validation

Our small dataset produced:

```text
Customer 1 → [F, O]
Customer 2 → [P]
Customer 3 → [F, O]
```

Therefore:

```python
array_contains(
    status_set,
    "P"
)
```

returned:

```text
Customer 2 only
```

This was an important validation because it proved that the filter can exclude groups.

---

## 10. Real-Data Validation

For the three TPC-H customers we examined, all three had at least one `P` status.

Therefore:

```python
array_contains(
    distinct_statuses,
    "P"
)
```

returned all three customers.

That does not mean the filter always returns every customer.

It happened because all three selected customers actually contained `P`.

---

## 11. Observed TPC-H Example

Customer:

```text
721180
```

S3.16 showed:

```text
F = 15
O = 20
P = 3

Total = 38
```

Therefore we predicted:

```text
list_size = 38
set_size  = 3
```

and:

```text
collect_set(status)
→ [F, O, P]
```

after sorting.

TPC-H order-status meanings:

```text
F → Fulfilled
O → Open
P → Partially Fulfilled
```

---

## 12. collect_list() vs DataFrame.collect()

These are completely different operations.

### collect_list()

```python
F.collect_list("status")
```

Aggregation function.

```text
Rows inside each group
        ↓
Array inside that group
```

Example:

```text
Customer 1
F
F
O

↓

Customer 1
[F, F, O]
```

---

### DataFrame.collect()

```python
df.collect()
```

Spark action.

```text
Distributed DataFrame result
        ↓
ALL rows
        ↓
Driver / Python
```

This can cause Driver-memory problems if the result is large.

---

## 13. Memory Considerations

### DataFrame.collect()

Main risk:

```text
Entire result
→ Driver
```

A large result can exhaust Driver memory.

### collect_list() / collect_set()

They do not bring the whole DataFrame directly to the Driver.

However, a single group can still create a very large aggregation state.

Example:

```text
groupBy(country)

One country
→ millions of transaction IDs

collect_list(transaction_id)
→ extremely large array
```

So large groups can still cause memory pressure during aggregation.

---

# 14. Quick Comparison

| Function | Duplicates | NULLs | Output |
|---|---|---|---|
| `collect_list()` | Keeps | Skips | Array |
| `collect_set()` | Removes | Skips | Array |
| `DataFrame.collect()` | Keeps rows | Keeps row data | Python `list[Row]` |

---

# 15. Related Interview Questions and Answers

### Q1. What is the difference between `collect_list()` and `collect_set()`?

`collect_list()` keeps duplicate non-NULL values, while `collect_set()` keeps only distinct non-NULL values. Both return Spark arrays.

---

### Q2. Do `collect_list()` and `collect_set()` preserve order?

No. Element ordering after aggregation/shuffle is not guaranteed.

Use:

```python
F.sort_array(...)
```

when stable value ordering is required.

---

### Q3. Does `sort_array()` restore the original row order?

No.

It sorts array values according to their value.

It does not know the original source-row ordering.

---

### Q4. What is the relationship between `size(collect_list(col))` and `count(col)`?

They should match because both consider non-NULL column values.

```text
size(collect_list(col))
=
count(col)
```

This may differ from:

```text
count("*")
```

when the column contains NULLs.

---

### Q5. What happens to NULL values?

NULL input values are not included in the collected array.

Example:

```text
F, NULL, O
```

becomes:

```text
[F, O]
```

---

### Q6. Is `collect_list()` the same as `DataFrame.collect()`?

No.

```text
collect_list()
→ aggregation function
→ array per group
```

while:

```text
DataFrame.collect()
→ action
→ all DataFrame result rows returned to Driver
```

---

### Q7. Which operation can cause Driver OutOfMemory errors?

`DataFrame.collect()` can exhaust Driver memory because it returns the entire result to the Driver.

`collect_list()` can instead create memory pressure inside an aggregation when one group contains a very large number of values.

---

### Q8. When should you use `collect_set()`?

When the business requirement needs the distinct values associated with each group.

Example:

```text
Customer
→ distinct order statuses
```

Result:

```text
721180
→ [F, O, P]
```

---

### Q9. How do you check whether a collected array contains a value?

Use:

```python
F.array_contains(
    F.col("status_set"),
    "P"
)
```

---

### Q10. Is `collect_list() → explode()` an exact reverse transformation?

No.

It can recover collected non-NULL values and their multiplicity, but NULLs, original row ordering and uncollected columns are not automatically restored.

---

## 16. Core Understanding

```text
Need all non-NULL values including duplicates?
→ collect_list()

Need distinct non-NULL values?
→ collect_set()

Need stable value ordering?
→ sort_array()

Need number of array elements?
→ size()

Need to test membership?
→ array_contains()

Need array elements back as rows?
→ explode()

Need all DataFrame rows in Python?
→ DataFrame.collect()
→ only when the result is known to be small
```

**S3.17 — collect_list() and collect_set() completed.**